# Notebook 1 — Dataset Preparation (one folder per breed)

Builds `data/{train,val,test}/<breed>/*.jpg` with **36 breed folders** total
(12 restricted + 24 unrestricted). A JSON manifest `breed_to_restricted.json`
records which breeds are restricted, so later notebooks can collapse
the multi-class softmax to a binary "restricted vs unrestricted" decision.

Idempotent: each step skips itself if its output is already correct.


In [ ]:
# ── Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── Verify GPU ────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('No GPU — go to Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# ── Project root on Drive ─────────────────────────────────
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/MSc_Capstone')
for d in ['data', 'pretrained/checkpoints', 'pretrained/logs',
          'pretrained/curves', 'pretrained/inference', 'report']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

DATA   = ROOT / 'data'
CKPT   = ROOT / 'pretrained' / 'checkpoints'
LOGS   = ROOT / 'pretrained' / 'logs'
CURVES = ROOT / 'pretrained' / 'curves'
INFER  = ROOT / 'pretrained' / 'inference'
REPORT = ROOT / 'report'

print(f'Project root: {ROOT}')
print(f'Data folder:  {DATA}')


## 1.1 — Download Stanford Dogs (skip if already done)


In [ ]:
import urllib.request

TAR_PATH   = DATA / 'images.tar'
IMAGES_DIR = DATA / 'images'
URL = 'http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar'

if not TAR_PATH.exists():
    print('Downloading Stanford Dogs (~800MB)...')
    urllib.request.urlretrieve(URL, TAR_PATH)
    print('Download complete.')
else:
    print('images.tar exists — skipping download.')


## 1.2 — Extract


In [ ]:
import tarfile

if not IMAGES_DIR.exists():
    capital_I = DATA / 'Images'
    if capital_I.exists():
        capital_I.rename(IMAGES_DIR)
    else:
        print('Extracting...')
        with tarfile.open(TAR_PATH, 'r') as tar:
            tar.extractall(DATA, filter='data')
        capital_I = DATA / 'Images'
        if capital_I.exists():
            capital_I.rename(IMAGES_DIR)
        print('Extraction complete.')
else:
    print('images/ exists — skipping extraction.')

print(f'Breeds found: {len(list(IMAGES_DIR.iterdir()))}')


## 1.3 — Normalise breed folder names


In [ ]:
import re, shutil

first_dir = next(IMAGES_DIR.iterdir())
already_normalised = not bool(re.match(r'^n[0-9]+-', first_dir.name))

if already_normalised:
    print('Already normalised — skipping.')
else:
    def normalise(breed_dir):
        slug = re.sub(r'^n[0-9]+-', '', breed_dir.name).lower().replace('-', '_')
        new_dir = breed_dir.parent / slug
        if breed_dir != new_dir:
            if new_dir.exists(): shutil.rmtree(new_dir)
            breed_dir.rename(new_dir)
            breed_dir = new_dir
        for i, img in enumerate(sorted(breed_dir.glob('*'))):
            if img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                img.rename(breed_dir / f'{slug}_{i+1:03d}{img.suffix.lower()}')
    for d in IMAGES_DIR.iterdir():
        if d.is_dir(): normalise(d)
    print(f'Normalised {len(list(IMAGES_DIR.iterdir()))} breeds.')

ALL_BREEDS = sorted(d.name for d in IMAGES_DIR.iterdir() if d.is_dir())
print(f'Total breeds: {len(ALL_BREEDS)}')


## 1.4 — Identify restricted breeds (keyword match)

Stanford folder names don't always match the legal breed names exactly
(`german_shepherd` vs `german_shepherd_dog`), so we match by keyword.


In [ ]:
RESTRICTED_KEYWORDS = [
    (['bull_mastiff', 'bullmastiff'],           'Bull Mastiff'),
    (['doberman'],                              'Doberman Pinscher'),
    (['german_shepherd'],                       'German Shepherd'),
    (['rhodesian_ridgeback'],                   'Rhodesian Ridgeback'),
    (['rottweiler'],                            'Rottweiler'),
    (['staffordshire_bull', 'staffordshire_bullterrier'], 'Staffordshire Bull Terrier'),
    (['american_pit', 'pit_bull'],              'American Pit Bull Terrier'),
    (['english_bull_terrier', 'bull_terrier'],  'English Bull Terrier'),
    (['japanese_akita', 'akita'],               'Japanese Akita'),
    (['japanese_tosa', 'tosa'],                 'Japanese Tosa'),
    (['bandog'],                                'Bandog'),
    (['xl_bully', 'american_bully'],            'XL Bully'),
]

RESTRICTED = set()
print('Restricted breeds present in Stanford Dogs:')
for keywords, display in RESTRICTED_KEYWORDS:
    matches = [b for b in ALL_BREEDS if any(k in b for k in keywords)]
    if matches:
        for m in matches:
            RESTRICTED.add(m)
            n = len(list((IMAGES_DIR / m).glob('*.jpg')))
            print(f'  {display:30s} -> {m} ({n} images)')
    else:
        print(f'  {display:30s} -> NOT FOUND')

print(f'\nTotal restricted breeds matched: {len(RESTRICTED)} of 12')


## 1.5 — Pick 24 unrestricted breeds (deterministic) and assemble the breed list

We use all available restricted breeds and randomly sample 24 unrestricted
breeds (with a fixed seed) so the experiment is reproducible. The full
breed list — 36 entries — is saved alongside `is_restricted` flags so
every notebook downstream uses the same mapping.


In [ ]:
import json, random

SEED = 58
random.seed(SEED)

pool = sorted(set(ALL_BREEDS) - RESTRICTED)
UNRESTRICTED = sorted(random.sample(pool, min(24, len(pool))))

SELECTED_BREEDS = sorted(RESTRICTED) + UNRESTRICTED
BREED_TO_RESTRICTED = {b: (b in RESTRICTED) for b in SELECTED_BREEDS}

manifest_path = DATA / 'breed_to_restricted.json'
with open(manifest_path, 'w') as f:
    json.dump({
        'breeds': SELECTED_BREEDS,
        'restricted': sorted(RESTRICTED),
        'unrestricted': UNRESTRICTED,
        'breed_to_restricted': BREED_TO_RESTRICTED,
        'seed': SEED,
    }, f, indent=2)

print(f'{len(SELECTED_BREEDS)} breeds selected ({len(RESTRICTED)} restricted + {len(UNRESTRICTED)} unrestricted).')
print(f'Manifest written: {manifest_path}')


## 1.6 — Build `data/{train,val,test}/<breed>/` (stratified 60/20/20)

Each breed becomes its own folder under each split. Stratification is
per-breed, so every breed appears in train, val, and test in the same
60/20/20 proportion. Re-running is safe: if the layout is already
correct it skips straight to verification.


In [ ]:
import shutil

SPLITS = ('train', 'val', 'test')

def layout_is_valid():
    for split in SPLITS:
        split_dir = DATA / split
        if not split_dir.is_dir(): return False
        existing = {d.name for d in split_dir.iterdir() if d.is_dir()}
        if existing != set(SELECTED_BREEDS):
            return False
        for breed in SELECTED_BREEDS:
            if not any((split_dir / breed).glob('*.jpg')):
                return False
    return True

if layout_is_valid():
    print('Splits already valid — skipping rebuild.')
else:
    # Wipe the old layout (binary or partial) before rebuilding.
    for legacy in ['restricted', 'unrestricted']:
        p = DATA / legacy
        if p.exists():
            print(f'Removing legacy {p}')
            shutil.rmtree(p)
    for split in SPLITS:
        p = DATA / split
        if p.exists():
            print(f'Removing old split: {p}')
            shutil.rmtree(p)
        for breed in SELECTED_BREEDS:
            (p / breed).mkdir(parents=True)

    def split_counts(n):
        t = int(n * 0.6); v = int(n * 0.2)
        return t, v, n - t - v

    random.seed(SEED)
    summary = {s: 0 for s in SPLITS}
    per_breed = {}

    for breed in SELECTED_BREEDS:
        imgs = sorted((IMAGES_DIR / breed).glob('*.jpg'))
        random.shuffle(imgs)
        t, v, te = split_counts(len(imgs))
        parts = {'train': imgs[:t], 'val': imgs[t:t+v], 'test': imgs[t+v:]}
        per_breed[breed] = {s: len(parts[s]) for s in SPLITS}
        for split, files in parts.items():
            for f in files:
                shutil.copy2(f, DATA / split / breed / f.name)
                summary[split] += 1

    print(f'\n{"Split":<6} | {"Images":>7}')
    print('-' * 18)
    for s in SPLITS:
        print(f'{s:<6} | {summary[s]:>7}')
    print(f'\nPer-breed counts (train/val/test):')
    for breed, counts in per_breed.items():
        tag = 'R' if BREED_TO_RESTRICTED[breed] else 'U'
        print(f'  [{tag}] {breed:30s} {counts["train"]:>4}/{counts["val"]:>3}/{counts["test"]:>3}')


## 1.7 — Final verification


In [ ]:
print('FINAL VERIFICATION\n')
ok = True
totals = {'restricted': 0, 'unrestricted': 0}
for split in SPLITS:
    r = u = 0
    for breed in SELECTED_BREEDS:
        p = DATA / split / breed
        if not p.is_dir():
            print(f'MISSING: {p}'); ok = False; continue
        n = len(list(p.glob('*.jpg')))
        if n == 0:
            print(f'EMPTY:   {p}'); ok = False
        if BREED_TO_RESTRICTED[breed]:
            r += n
        else:
            u += n
    totals['restricted'] += r
    totals['unrestricted'] += u
    print(f'{split.upper():5s}  restricted={r:>4}  unrestricted={u:>4}  total={r+u:>4}')

print(f'\nOverall: restricted={totals["restricted"]}, unrestricted={totals["unrestricted"]}')
print('ALL CHECKS PASSED.' if ok else 'PROBLEMS DETECTED — check above.')
if ok:
    print('\nNotebook 1 complete. Open Notebook 2 and Run all.')
